# 🎯 Stack 2.9 — 128K Context Fine-tuning

Fine-tune **Qwen2.5-Coder-1.5B** with **packed 128K context windows**.

**Key innovation:** Instead of training on short ~500-token examples, we **pack 200+ examples** into each 128K window. This multiplies training signal and teaches the model to track tool state across long, multi-turn interactions.

**Runtime:** Runtime → Change runtime type → **GPU (T4 16GB recommended)**
**Time:** ~6-8 hours on free Colab T4

## Step 1: Clone Stack 2.9 & Install Dependencies

In [ ]:
# Clone the repo (gets the fixed training script)
!git clone https://github.com/my-ai-stack/stack-2.9.git
cd stack-2.9

# Install all dependencies
!pip install -q transformers peft datasets bitsandbytes>=0.46.1 accelerate huggingface_hub scipy
!pip install -q torch --upgrade

print('✅ Dependencies installed')

## Step 2: Login to HuggingFace

Get your token at: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
# 👇 Replace with YOUR HuggingFace token
login(token="YOUR_HF_TOKEN_HERE")  # ← 🔴 PUT YOUR HF TOKEN HERE
print('✅ Logged into HuggingFace')

## Step 3: Mount Google Drive

Training checkpoints and the final adapter will be saved here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/stack-2.9-128k-output'
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'📁 Output directory: {OUTPUT_DIR}')

## Step 4: Download Training Data

We use the dataset uploaded to HuggingFace Hub — 1500 tool-calling examples, packed into 128K sequences.

In [ ]:
import huggingface_hub

DATA_FILE = '/content/tool_examples.jsonl'

print('Downloading training data from HuggingFace...')
hf_id = 'walidsobhie/stack-2-9-tool-examples'
path = huggingface_hub.hf_hub_download(
    repo_id=hf_id,
    filename='tool_examples_combined.jsonl',
    repo_type='dataset',
    local_dir='/content/',
    local_dir_use_symlinks=False,
)
import shutil
shutil.move(path, DATA_FILE)
print(f'✅ Dataset ready: {DATA_FILE}')

# Quick sanity check
import json
with open(DATA_FILE) as f:
    lines = f.readlines()
print(f'   Total examples: {len(lines)}')
ex = json.loads(lines[0])
print(f'   Keys: {list(ex.keys())}')

## Step 5: Run 128K Packed Context Fine-tuning

**This cell runs the full training. On free Colab T4 it takes ~6-8 hours.**

If Colab disconnects, your checkpoints are safe in Google Drive. Reconnect and re-run this cell — it will resume from the last checkpoint.

In [ ]:
import subprocess

# Run the fixed training script with packing enabled
result = subprocess.run([
    "python3", "training/train_extended_context.py",
    "--model-path", "Qwen/Qwen2.5-Coder-1.5B",
    "--data-path", "/content/tool_examples.jsonl",
    "--output-dir", OUTPUT_DIR,
    "--context-length", "131072",
    "--lora-rank", "32",
    "--epochs", "3",
    "--batch-size", "1",
    "--grad-accum", "16",
    "--lr", "2e-4",
    "--use-packing",
    "--push-to-hub",
    "--hub-model-id", "walidsobhie/stack-2.9-128k-context"
], cwd="/content/stack-2.9")

print('STDOUT:', result.stdout)
print('STDERR:', result.stderr[-3000:] if result.stderr else '(none)')